In [1]:
!pip install kaggle==1.5.16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for kaggle: filename=kaggle-1.5.16-py3-none-any.whl size=110682 sha256=7e10290a3a1425f1a972477c70c417480b28f9a3e0304a91e025e1646c082711
  Stored in directory: /root/.cache/pip/wheels/6a/2e/62/475f9443c6f7f73b3beb46e121e2d30f1fb77af8bc7ba7edd6
Successfully built kaggle
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


In [2]:
! chmod 600 .kaggle/kaggle.json

chmod: cannot access '.kaggle/kaggle.json': No such file or directory


In [3]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

  0% 0.00/2.70M [00:00<?, ?B/s]
100% 2.70M/2.70M [00:00<00:00, 190MB/s]


In [4]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

Archive:  Walmart-Recruiting-Store-Sales-Forecasting.zip
  inflating: features.csv.zip        
  inflating: sampleSubmission.csv.zip  
  inflating: stores.csv              
  inflating: test.csv.zip            
  inflating: train.csv.zip           


In [5]:
! unzip features.csv.zip

Archive:  features.csv.zip
  inflating: features.csv            


In [6]:
! unzip test.csv.zip

Archive:  test.csv.zip
  inflating: test.csv                


In [7]:
! unzip train.csv.zip

Archive:  train.csv.zip
  inflating: train.csv               


# N-BEATS — Walmart Store Sales Forecasting

In [11]:
!pip install "numpy<2" "torchvision==0.17.0" "torch==2.2.0" "neuralforecast==1.7.4" optuna mlflow dagshub wandb -q --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import wandb
import mlflow
import mlflow.pyfunc
import optuna

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.auto import AutoNBEATS
from neuralforecast.losses.pytorch import MAE

optuna.logging.set_verbosity(optuna.logging.WARNING)

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'NBEATS'

MLFLOW_EXPERIMENT = 'NBEATS_Training'
MLFLOW_MODEL_NAME = 'nbeats_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4    # forecast horizon (weeks)
INPUT_SIZE = 52   # lookback window (1 year)
N_TRIALS   = 5   # Optuna trials
VAL_START  = '2012-04-01'

wandb.login()
# mlflow.set_experiment(MLFLOW_EXPERIMENT)

print('Setup OK')

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ikvas22 (ikvas22-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setup OK


## 1. Pre-processing

In [2]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'NBEATS_Preprocessing',
)

# ── Load & merge ───────────────────────────────────────────────
train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows'  : df.shape[0],
    'raw_cols'  : df.shape[1],
    'stores'    : df['Store'].nunique(),
    'depts'     : df['Dept'].nunique(),
    'date_min'  : str(df['Date'].min().date()),
    'date_max'  : str(df['Date'].max().date()),
})

# ── Null check ─────────────────────────────────────────────────
null_pct = df.isnull().mean().mul(100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['column', 'null_pct']
wandb.log({'null_percentages': wandb.Table(dataframe=null_df)})
print('Nulls (%):')
print(null_df.to_string(index=False))

# ── Format for neuralforecast: unique_id | ds | y ──────────────
# N-BEATS only needs the raw sales series — no tabular features
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df_nf = (
    df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

# ── Drop series too short for the lookback window ──────────────
min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'    : len(series_len),
    'valid_series'    : len(valid_ids),
    'dropped_series'  : len(dropped),
    'dropped_ids'     : dropped,
    'min_series_len'  : int(series_len[valid_ids].min()),
    'max_series_len'  : int(series_len[valid_ids].max()),
})

print(f'\nValid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

# ── Train / val split ──────────────────────────────────────────
df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'train_date_min' : str(df_train['ds'].min().date()),
    'train_date_max' : str(df_train['ds'].max().date()),
    'val_date_min'   : str(df_val['ds'].min().date()),
    'val_date_max'   : str(df_val['ds'].max().date()),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'lookback_weeks' : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

Nulls (%):
   column  null_pct
MarkDown1     64.26
MarkDown2     73.61
MarkDown3     67.48
MarkDown4     67.98
MarkDown5     64.08

Valid series : 2983
Dropped      : 348 (shorter than 56 weeks)
Train : 2010-02-05 → 2012-03-30  (328,806 rows)
Val   : 2012-04-06 → 2012-10-26  (87,383 rows)


depts,▁
dropped_series,▁
horizon_weeks,▁
lookback_weeks,▁
max_series_len,▁
min_series_len,▁
raw_cols,▁
raw_rows,▁
stores,▁
total_series,▁
+3,...


## 2. Training

In [3]:
# WMAE — the official Walmart competition metric
# Holiday weeks are weighted 5x more than regular weeks
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def evaluate(nf: NeuralForecast, pred_col: str) -> tuple:
    """Run prediction on val set and return (wmae, mae, eval_df)."""
    preds   = nf.predict()
    eval_df = df_val.merge(
        preds.rename(columns={pred_col: 'y_pred'}),
        on=['unique_id', 'ds'], how='inner'
    )
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [4]:
# ── Baseline: single N-BEATS with sensible defaults ────────────
# Run this first to verify the pipeline works and get a reference WMAE
# before spending time on hyperparameter search
run_baseline = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'NBEATS_Baseline',
    config   = {
        'input_size'    : INPUT_SIZE,
        'h'             : H,
        'stack_types'   : ['trend', 'seasonality'],
        'n_blocks'      : [3, 3],
        'mlp_units'     : [[512, 512], [512, 512]],
        'n_harmonics'   : 2,
        'n_polynomials' : 2,
        'max_steps'     : 200,
        'learning_rate' : 1e-3,
        'batch_size'    : 32,
        'loss'          : 'MAE',
        'variant'       : 'interpretable',
    }
)

baseline_model = NBEATS(
    h               = H,
    input_size      = INPUT_SIZE,
    stack_types     = ['trend', 'seasonality'],
    n_blocks        = [3, 3],
    mlp_units       = [[512, 512], [512, 512]],
    n_harmonics     = 2,
    n_polynomials   = 2,
    loss            = MAE(),
    max_steps       = 500,
    learning_rate   = 1e-3,
    batch_size      = 32,
    val_check_steps = 50,
    logger          = True,
)

nf_baseline = NeuralForecast(models=[baseline_model], freq='W-FRI')
nf_baseline.fit(df=df_train, val_size=len(df_val['ds'].unique()))

baseline_wmae, baseline_mae, eval_baseline = evaluate(nf_baseline, 'NBEATS')

wandb.log({
    'val_wmae' : baseline_wmae,
    'val_mae'  : baseline_mae,
})

print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Baseline MAE  : {baseline_mae:,.2f}')

wandb.finish()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  3.3 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.3 M                                                                                            
Non-trainable params: 1.5 K                                                                                        
Total params: 3.3 M                                                                                                
Total estimated model params size (MB): 13.376                                                                     
Modules in train mode: 52                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Baseline WMAE : 1,990.94
Baseline MAE  : 1,990.94


val_mae,▁
val_wmae,▁
val_mae,1990.94296
val_wmae,1990.94296


In [ ]:
# ── AutoNBEATS: Optuna hyperparameter search ───────────────────
# Searches over the most impactful N-BEATS hyperparameters.
# Each trial = one full training run evaluated on val set.
# This is the DL equivalent of GridSearchCV.
run_search = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'NBEATS_HPSearch',
    config   = {
        'n_trials'    : N_TRIALS,
        'h'           : H,
        'search_space': {
            'input_size'    : [26, 52, 104],
            'n_blocks'      : '1–5 per stack',
            'mlp_units'     : [128, 256, 512],
            'n_harmonics'   : '1–4',
            'n_polynomials' : '1–3',
            'learning_rate' : '1e-4 – 1e-3 (log)',
            'batch_size'    : [16, 32, 64],
        }
    }
)

from ray import tune

nbeats_config = {
    'input_size'    : tune.choice([26, 52, 104]),
    'n_blocks'      : tune.choice([[1,1], [2,2], [3,3], [4,4], [5,5]]),
    'mlp_units'     : tune.choice([
                        [[128,128],[128,128]],
                        [[256,256],[256,256]],
                        [[512,512],[512,512]],
                      ]),
    'n_harmonics'   : tune.randint(1, 5),
    'n_polynomials' : tune.randint(1, 4),
    'learning_rate' : tune.loguniform(1e-4, 1e-3),
    'batch_size'    : tune.choice([16, 32, 64]),
    'max_steps'     : 500,
}

auto_model = AutoNBEATS(
    h           = H,
    config      = nbeats_config,
    num_samples = N_TRIALS,
    loss        = MAE(),
)

nf_auto = NeuralForecast(models=[auto_model], freq='W-FRI')
nf_auto.fit(df=df_train, val_size=len(df_val['ds'].unique()))

best_wmae, best_mae, eval_best = evaluate(nf_auto, 'AutoNBEATS')
best_params = auto_model.results.get_best_result().config
trials_df   = auto_model.results.get_dataframe()

wandb.log({
    'best_val_wmae'    : best_wmae,
    'best_val_mae'     : best_mae,
    'baseline_wmae'    : baseline_wmae,
    'wmae_improvement' : baseline_wmae - best_wmae,
    'best_params'      : str(best_params),   # str() because wandb can't log dicts with tune objects
    'trials_completed' : N_TRIALS,
    'all_trials'       : wandb.Table(dataframe=trials_df),
})

print(f'Best WMAE     : {best_wmae:,.2f}')
print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Improvement   : {baseline_wmae - best_wmae:,.2f}')
print(f'Best params   : {best_params}')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='N-BEATS', color='tomato', linestyle='--')
    hol = actual[actual['IsHoliday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('N-BEATS Best Model — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

# Per-series WMAE
per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['IsHoliday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

+--------------------------------------------------------------------+
| Configuration for experiment     _train_tune_2026-06-28_17-32-58   |
+--------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator             |
| Scheduler                        FIFOScheduler                     |
| Number of trials                 20                                |
+--------------------------------------------------------------------+

View detailed results here: /root/ray_results/_train_tune_2026-06-28_17-32-58
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-06-28_17-29-46_441808_6129/artifacts/2026-06-28_17-32-58/_train_tune_2026-06-28_17-32-58/driver_artifacts`


## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
# Wrapper so the registered model accepts raw unprocessed Walmart data
# and can be called directly in model_inference.ipynb without any preprocessing
class NBEATSWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['nf_model'], 'rb') as f:
            self.nf = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Returns [Store, Dept, Date, Weekly_Sales_pred].
        """
        df_in              = model_input.copy()
        df_in['Date']      = pd.to_datetime(df_in['Date'])
        df_in['unique_id'] = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)
        df_nf_in = (
            df_in
            .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            [['unique_id', 'ds', 'y']]
            .sort_values(['unique_id', 'ds'])
        )
        preds    = self.nf.predict(df=df_nf_in)
        pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
        preds    = preds.rename(columns={pred_col: 'Weekly_Sales_pred'})
        preds[['Store', 'Dept']] = preds['unique_id'].str.split('_', expand=True).astype(int)
        return preds[['Store', 'Dept', 'ds', 'Weekly_Sales_pred']].rename(columns={'ds': 'Date'})


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/nf_auto_nbeats.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(nf_auto, f)

with mlflow.start_run(run_name='NBEATS_Best_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'val_wmae'         : best_wmae,
        'val_mae'          : best_mae,
        'baseline_wmae'    : baseline_wmae,
        'wmae_improvement' : baseline_wmae - best_wmae,
        'n_trials'         : N_TRIALS,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'nbeats_model',
        python_model          = NBEATSWrapper(),
        artifacts             = {'nf_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
# Sanity check: load from registry and predict on raw data
# This is exactly how model_inference.ipynb will use it
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
sample = train_raw[train_raw['Store'] == 1][['Store', 'Dept', 'Date', 'Weekly_Sales']].head(100)
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())